# Husky 와이어태핑을 활용한 능동적 오류주입 공격 (Active Wiretapping for FA)

## 두 ChipWhisperer 장치의 역할 분리 실험 — Lite (정상 호스트) + Husky (능동적 공격자)

---

### 🎯 노트북의 목표

이 노트북은 기존의 **수동적 부채널 와이어태핑(Wiretapping4SCA)** 시나리오를 확장하여, **능동적 오류주입 공격(Fault Injection)** 환경을 구축하는 방법을 다룹니다. 

실제 공격 시나리오와 유사하게, **CW-Lite는 대상 기기(Target)와 정상적으로 통신하는 호스트 역할**만 수행하며 자신이 공격받고 있다는 사실을 모릅니다. 반면 **Husky는 두 기기 사이의 통신/클럭 라인을 와이어태핑**하고, 적절한 타이밍(Trigger)을 가로채어 타겟의 클럭에 **치명적인 글리치(Glitch)를 주입하는 공격자 역할**을 수행합니다.

### 🔌 하드웨어 결선 가이드 (매우 중요)

이 시나리오는 하드웨어가 올바르게 연결되어 있어야 작동합니다.

1. **Lite ↔ Target (정상 통신로):** 기존과 동일하게 20-pin 리본 케이블을 연결합니다. (전원, 클럭, 트리거, Tx, Rx 공급)
2. **Husky ↔ Target (와이어태핑 및 공격로):**
   * **Clock/Trigger Sync:** Target의 Clock 및 Trigger 핀을 Husky의 `HS1` / `HS2` 또는 전면 I/O에 점퍼선으로 연결하여 신호를 훔쳐옵니다(Wiretapping).
   * **Glitch Injection:** Husky의 `Glitch 포트(SMA)`를 Target의 VCC 또는 Clock 라인(Glitch 입력단)에 물리적으로 연결합니다.

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 다중 장치(Lite + Husky) 동시 연결 및 역할 부여 | `lite_scope`, `husky_scope` |
| **2단계** | 정상 호스트(Lite) ↔ 타겟 통신 채널 확보 | `target` 객체, 펌웨어 플래싱 |
| **3단계** | 공격자(Husky)의 신호 동기화 및 글리치 모듈 설정 | `husky_scope.glitch` 설정 완료 |
| **4단계** | 광범위 파라미터 탐색 (Fault Injection Campaign) | 3중 루프 실행, `cglitch_result` |
| **5단계** | 공격 결과 시각화 및 최적 파라미터 도출 | Bokeh 산점도 시각화 |


In [ ]:
import chipwhisperer as cw
import time
import numpy as np
from tqdm.notebook import trange

# Bokeh 시각화 도구
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.palettes import Category10
import scipy.stats as sp
output_notebook()

### 1단계: 장치 식별 및 동시 연결

Lite와 Husky를 동시에 제어하기 위해 Serial Number(SN)를 사용하여 명시적으로 연결합니다.
`cw.list_devices()`를 실행하여 각 장치의 SN을 확인하고 아래 코드에 입력하세요.

In [ ]:
# 현재 연결된 기기 목록 확인 (주석 해제 후 실행)
# print(cw.list_devices())

# TODO: 각 연구자의 환경에 맞게 SN을 수정하세요.
SN_LITE = "442031204630..."  # CW-Lite (Host)
SN_HUSKY = "502031203841..." # CW-Husky (Attacker)

try:
    if not lite_scope.connectStatus:
        pass
except NameError:
    print("[Lite] 정상 호스트 장치 연결 중...")
    lite_scope = cw.scope(sn=SN_LITE)
    
try:
    if not husky_scope.connectStatus:
        pass
except NameError:
    print("[Husky] 공격자 장치 연결 중...")
    husky_scope = cw.scope(sn=SN_HUSKY)

print("✅ 두 장치 모두 연결 완료!")

### 2단계: 정상 호스트(Lite) 설정 및 타겟 프로그래밍

CW-Lite는 Target 보드에 클럭을 공급하고 통신을 담당합니다. 타겟 보드를 초기화하고 간단한 루프 연산을 수행하는 펌웨어를 업로드합니다. (기존 FA_main과 동일한 타겟 코드 사용 가정)

In [ ]:
# Lite를 통신 컨트롤러로 설정
target = cw.target(lite_scope, cw.targets.SimpleSerial2)

# Lite 클럭 및 통신 설정 (표준 설정)
lite_scope.default_setup()

print("🔧 타겟 펌웨어 컴파일 및 업로드 (Lite 활용)...")
cw.program_target(lite_scope, cw.programmers.STM32FProgrammer, "../hardware/victims/firmware/simpleserial-glitch/simpleserial-glitch-CW308_STM32F3.hex")

time.sleep(0.5)
print("✅ 타겟 통신/클럭 설정 완료 (Lite)")

### 3단계: 공격자(Husky) 와이어태핑 동기화 및 글리치 설정

**이 단계가 이 시나리오의 핵심입니다.** 
Husky는 내부 클럭을 사용하지 않고, Target으로 들어가는 Lite의 클럭 신호(`extclk_aux_io`)를 도청하여 자신의 클럭을 완벽히 동기화(PLL 잠금)합니다. 그 후 도청된 Trigger 신호를 바탕으로 Glitch 모듈을 활성화합니다.

In [ ]:
# 3.1 클럭 와이어태핑 (Husky 동기화)
# Husky의 Target IO IN 핀으로 들어오는 외부 클럭을 소스로 사용합니다.
husky_scope.clock.clkgen_src = 'extclk_aux_io'
time.sleep(0.5)

# 주파수 확인 및 PLL 동기화
ext_freq = husky_scope.clock.extclk_freq
print(f"[Husky] 감지된 외부 클럭 주파수: {ext_freq/1e6:.2f} MHz")

husky_scope.clock.clkgen_freq = ext_freq # 발견한 클럭으로 주파수 락(Lock)
husky_scope.clock.adc_mul = 1 # 1 sample = 1 clock (FA의 기본)

# 3.2 트리거 와이어태핑
# Target이 통신을 시작할 때 발생하는 Trigger 신호(tio4)를 가로챕니다.
husky_scope.trigger.triggers = 'tio4'

# 3.3 글리치 모듈 설정 (Clock Glitch 기준)
husky_scope.glitch.clk_src = 'pll'           # 동기화된 클럭 기반
husky_scope.glitch.output = 'clock_xor'      # 기존 클럭에 XOR하여 파형을 왜곡
husky_scope.glitch.trigger_src = 'ext_single' # 가로챈 트리거 1회에 반응

print("✅ Husky 와이어태핑 및 글리치 모듈 준비 완료!")

### 4단계: 오류주입 파라미터 탐색 (Fault Injection Campaign)

FA_main에서 학습한 3중 루프(`ext_offset`, `offset`, `width`)를 수행합니다.
여기서 중요한 점은 **타겟 조작은 Lite가(`target.simpleserial_write`), 공격 준비는 Husky가(`husky_scope.arm()`)** 담당한다는 것입니다.

In [ ]:
# 타겟을 안전하게 재부팅하기 위한 헬퍼 함수 (Lite를 통해 전원/리셋 제어)
def reboot_target():
    lite_scope.io.nrst = 'low'
    time.sleep(0.05)
    lite_scope.io.nrst = 'high_z'
    time.sleep(0.05)

# 정상 응답값 확인 (Lite를 통해 요청)
reboot_target()
target.simpleserial_write('g', bytearray([]))
expected_ret = target.simpleserial_read('r', 4)
print(f"🎯 정상 동작 시 반환값 (expected_ret): {expected_ret}")

# 탐색 범위 설정 (강의용 축소 범위)
ext_offsets = range(10, 30, 2)  # 글리치 시점 (클럭 단위)
offsets = np.arange(-40, 40, 5) # 클럭 내부 위상
widths = np.arange(-40, 40, 5)  # 글리치 폭

results = []

print("🚀 공격 캠페인 시작...")
for ext_offset in trange(len(ext_offsets), desc="Ext Offset 탐색"):
    husky_scope.glitch.ext_offset = ext_offsets[ext_offset]
    
    for offset in offsets:
        husky_scope.glitch.offset = offset
        
        for width in widths:
            husky_scope.glitch.width = width
            
            # 1. 대상 통신 버퍼 초기화
            target.flush()
            
            # 2. 공격자(Husky) 장전 - 트리거를 기다림
            husky_scope.arm()
            
            # 3. 호스트(Lite)가 정상 명령 전송 -> 이 순간 Trigger가 발생하며 Husky가 글리치 주입
            target.simpleserial_write('g', bytearray([]))
            
            # 4. 결과 읽기 및 분류
            ret = target.simpleserial_read('r', 4)
            
            if ret is None: # 응답 없음 (칩 다운됨)
                status = "reset"
                reboot_target()
            elif ret == expected_ret: # 정상 응답 (공격 실패)
                status = "normal"
            else: # 비정상 응답 (오류 주입 성공!!)
                status = "success"
                print(f"🎉 SUCCESS at Ext_off:{husky_scope.glitch.ext_offset}, Off:{offset}, Wid:{width} | Ret: {ret}")
                
            # 결과 저장
            results.append((husky_scope.glitch.ext_offset, offset, width, status))

print("✅ 공격 캠페인 완료!")

### 5단계: 공격 결과 시각화 (Bokeh)

수집된 결과를 바탕으로, 어느 `(offset, width)` 파라미터 조합에서 성공적인 공격(Success)과 시스템 리셋(Reset)이 발생하는지 분포를 확인합니다.

In [ ]:
# 결과 데이터 정리
data_normal = {'x': [], 'y': []}
data_success = {'x': [], 'y': []}
data_reset = {'x': [], 'y': []}

for ext_off, off, wid, status in results:
    if status == 'normal':
        data_normal['x'].append(wid)
        data_normal['y'].append(off)
    elif status == 'success':
        data_success['x'].append(wid)
        data_success['y'].append(off)
    elif status == 'reset':
        data_reset['x'].append(wid)
        data_reset['y'].append(off)

# Bokeh 플롯 생성
p = figure(title="Fault Injection Results (Husky Wiretapping)", x_axis_label="Glitch Width", y_axis_label="Glitch Offset")

p.scatter(data_normal['x'], data_normal['y'], color="green", legend_label="Normal", alpha=0.1, size=5)
p.scatter(data_reset['x'], data_reset['y'], color="red", legend_label="Reset (Crash)", alpha=0.5, size=5)
p.scatter(data_success['x'], data_success['y'], color="blue", legend_label="Success (Fault!)", marker="star", size=12)

p.legend.click_policy = "hide"
show(p)

### 📝 요약 및 핵심 개념

| 구분 | 설명 |
|:---|:---|
| **장치 분리** | CW-Lite는 타겟 제어 및 정상 통신만 수행하며, CW-Husky는 도청과 타격(글리치)을 담당합니다. |
| **신호 동기화** | 공격자인 Husky가 정확한 시점에 타격하기 위해 대상의 클럭(`extclk_aux_io`)과 트리거(`tio4`)를 와이어태핑하여 PLL을 일치시키는 것이 필수적입니다. |
| **제어권 분리** | 루프 내에서 `target.write()`는 Lite가, `scope.arm()`은 Husky가 수행하는 비동기적 협업 패턴을 이해해야 합니다. |

실험이 끝난 후에는 장치를 안전하게 해제합니다.

In [ ]:
try:
    lite_scope.dis()
    husky_scope.dis()
    target.dis()
    print("✅ 모든 장치 안전하게 해제 완료.")
except:
    pass